[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Boolean Filtering


## What you will be able to do

Keep only the rows of a table that meet a condition, combine conditions on several columns, and
count the rows that match, without the precedence mistake or the missing readings changing the
answer.

## The idea

### The problem

Most questions about a table are about part of it. Which hours at Oslo were warmer than 15 degrees?
How many hours did any station spend below -15 or above 20? What happened at Bergen at 15:00 every
day in July? Each of those picks out some rows of the 35,040 and leaves the rest.

A loop can do it, one row at a time, with an `if` that tests the station, the temperature and the
hour and appends the rows that pass to a new list. The loop has to know where each value sits in the
row, it builds a second copy of the data by hand, and it walks all 35,040 rows in Python, which is
the slow side of the timings in the **Performance** notebook of the **NumPy, Deep Dive** guide. The
question you actually have, "Oslo and above 15", is a small part of the code.

The **Boolean Masks** notebook in the **NumPy, Deep Dive** guide answered questions like these for
an array. A table adds two things. A condition names a column instead of a position, and the rows
that pass keep their labels, so a filtered table still says which hour and which station each row
is.

### What a boolean mask is

> A **boolean mask** is a Series of `True` and `False` values with the same index as the table it
> was made from, usually by comparing a column with a value. Indexing the table with the mask,
> `readings[mask]`, keeps the rows whose label has `True` and drops the others.

### Why it works that way

A comparison between a Series and a value runs on every value at once and returns one `True` or
`False` per row, labeled like the column it came from. Putting that Series in square brackets lines
it up with the table by label and keeps the rows marked `True`. The rows that survive keep their
original index labels, which is why a filtered table's index has gaps.

Conditions combine with `&` for and, `|` for or and `~` for not. Python's own `and`, `or` and `not`
ask a single question of their operands, true or false, and a Series of 35,040 values has no single
answer, so pandas raises `ValueError` for them. The operators that do work bind more tightly than
`>` and `==`, so every condition needs its own parentheses, and leaving them out produces two
different errors, both in Common errors.

Missing values change the count. Every comparison with `NaN` is false except `!=`, which is true, so
Svalbard's 24 silent hours are neither below freezing nor at or above it, and a filter written with
`!=` keeps them. A filter drops them without saying so, and two filters that ask the same question
in different words can disagree by those 24 rows.

### Where this shows up

Every analysis filters. A report picks one station or one month, a cleaning step drops readings that
are impossible, and a chart shows only the rows that matter. The `WHERE` clause in the **sqlite3,
Deep Dive** guide does the same job inside a database. Later notebooks filter before they group,
merge or reshape, and **The Index** notebook selects rows by label instead.

### What this notebook covers

- a comparison as a mask, and the table filtered by it
- conditions combined with `&`, `|` and `~`
- `isin` for a list of values, and `between` for a range
- conditions on the time of a reading
- counting the rows a mask keeps, and the fraction it keeps
- `query`, which writes the same conditions as text
- four errors, two of them from a missing pair of parentheses

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import pandas as pd

readings = pd.DataFrame({
    "station": ["Bergen", "Oslo", "Bergen", "Oslo"],
    "celsius": [8.2, -1.5, 7.9, 2.4],
})
cold = readings["celsius"] < 3
print(cold.tolist())
print(readings[cold])
```

```
[False, True, False, True]
  station  celsius
1    Oslo     -1.5
3    Oslo      2.4
```

The comparison gave one `True` or `False` for each row, and the table indexed with it kept rows 1
and 3. They kept their labels, so the filtered table still says which rows of the original they
were.

## Setup

One library and three helpers for building the readings.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`readings` is the table of the four stations' hourly readings for 2025, 35,040 rows with the columns
`station`, `time` and `celsius`, made by the same formula as the other notebooks in this guide.
Svalbard's 24 readings for 2 March are `NaN`. `stations` holds each station's latitude, Kirkenes
included.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### A comparison makes a mask

Comparing the `celsius` column with 15 gives a Series of the same length, one `True` or `False` per
row:

In [2]:
warm = readings["celsius"] > 15

print(warm.dtype, len(warm))
print(warm.head(3))
print("hours above 15 degrees:", warm.sum())

bool 35040
0    False
1    False
2    False
Name: celsius, dtype: bool
hours above 15 degrees: 2804


The mask has the table's index, 0 to 35039, and the type `bool`. `sum` counts the `True` values,
because `True` counts as 1 and `False` as 0, so 2,804 of the hours were warmer than 15 degrees.

### Filtering the table with the mask

In [3]:
readings[warm]

,station,time,celsius
12732,Bergen,2025-05-13 15:00:00,15.1
12736,Bergen,2025-05-13 16:00:00,15.3
12824,Bergen,2025-05-14 14:00:00,15.2
12828,Bergen,2025-05-14 15:00:00,15.7
12916,Bergen,2025-05-15 13:00:00,15.2
...,...,...,...
25312,Bergen,2025-09-21 16:00:00,15.3
25316,Bergen,2025-09-21 17:00:00,15.3
25404,Bergen,2025-09-22 15:00:00,15.4
25408,Bergen,2025-09-22 16:00:00,15.6


The filtered table has 2,804 rows, and their index labels are the ones they had in `readings`, which
is why the first is 12732. The first warm hour of the year was Bergen at 15:00 on 13 May. Counting
the stations shows who had the warm hours:

In [4]:
readings[warm]["station"].value_counts()

station
Bergen    1621
Oslo      1043
Tromso     140
Name: count, dtype: int64

Svalbard is missing from the counts, because it never passed 15 degrees, and `value_counts` lists
only the values it found.

### Combining conditions with &, | and ~

`&` keeps the rows where both conditions are `True`. Each condition goes in its own parentheses:

In [5]:
oslo_warm = readings[(readings["station"] == "Oslo") & (readings["celsius"] > 15)]

print(len(oslo_warm), "warm hours at Oslo")

1043 warm hours at Oslo


All 1,043 of Oslo's warm hours are there, the same count `value_counts` gave above. `|`
keeps a row where either condition is `True`, here the hours below -15 or above 20:

In [6]:
extreme = readings[(readings["celsius"] < -15) | (readings["celsius"] > 20)]

extreme.groupby("station")["celsius"].agg(["count", "min", "max"])

,count,min,max
station,,,
Bergen,53,20.1,20.8
Svalbard,411,-17.3,-15.1


Bergen's 53 hours were all above 20 and Svalbard's 411 all below -15. `~` turns every `True`
into `False` and back, so it keeps the rows a condition rejects:

In [7]:
not_svalbard = readings[~(readings["station"] == "Svalbard")]

print(len(not_svalbard), "readings from the other three stations")
print("same mask as != :", (readings["station"] != "Svalbard").equals(~(readings["station"] == "Svalbard")))

26280 readings from the other three stations
same mask as != : True


`readings["station"] != "Svalbard"` makes the same mask, as the last line shows. `~` is useful when
a condition is long or already has a name. `~warm` keeps every row that `warm` dropped, and Common
errors shows what that does to the missing readings.

### A list of values with isin, and a range with between

`isin` is `True` where the value is any of the ones listed, which replaces a chain of `|`:

In [8]:
west_and_east = readings[readings["station"].isin(["Bergen", "Oslo"])]

print(len(west_and_east), "readings from Bergen and Oslo")

17520 readings from Bergen and Oslo


`between` is `True` where the value lies in a range, and includes both ends unless told otherwise:

In [9]:
near_freezing = readings["celsius"].between(-0.5, 0.5)

print("hours within half a degree of freezing:", near_freezing.sum())
print("0 to 0.5, both ends:", readings["celsius"].between(0, 0.5).sum())
print("0 to 0.5, without 0.5:", readings["celsius"].between(0, 0.5, inclusive="left").sum())

hours within half a degree of freezing: 1792
0 to 0.5, both ends: 1005
0 to 0.5, without 0.5: 845


The last two counts differ by the 160 readings of exactly 0.5 degrees. `inclusive` takes `"both"`,
the default, `"left"`, `"right"` or `"neither"`.

### Conditions on the time of a reading

`time` holds dates and times, and its `.dt` accessor gives each part as a column of numbers to
compare, which the **Dates and Times** notebook covers. Bergen at 15:00 on every day of July:

In [10]:
bergen_july_afternoons = readings[
    (readings["station"] == "Bergen")
    & (readings["time"].dt.month == 7)
    & (readings["time"].dt.hour == 15)
]

print(len(bergen_july_afternoons), "afternoons")
print(bergen_july_afternoons["celsius"].max(), "degrees at the warmest")

31 afternoons
20.8 degrees at the warmest


Inside square brackets a long condition can run over several lines, and putting each `&` at the
start of a line makes the conditions easy to read one at a time. July has 31 days, so there are 31
rows, and the warmest of them is 20.8 degrees, the highest reading of the year, which the **Why
Pandas** notebook found with `nlargest`.

### Counting and measuring with a mask

A mask's `sum` counts its `True` values, and its `mean` is the fraction that are `True`. Grouping a
mask by station gives each station's fraction of hours below freezing:

In [11]:
frost = readings["celsius"] < 0

print("hours below freezing:", frost.sum(), "of", len(frost))
readings.assign(frost=frost).groupby("station")["frost"].mean().round(3)

hours below freezing: 12142 of 35040


station
Bergen      0.139
Oslo        0.211
Svalbard    0.670
Tromso      0.366
Name: frost, dtype: float64

`assign` adds the mask as a column named `frost`, which the **Adding and Changing Columns** notebook
covers, so that `groupby` can average it for each station. Svalbard spent 0.67 of the year below
freezing. Its 24 silent hours are in that fraction as `False`, because `NaN < 0` is `False`, so the
fraction is of all 8,760 hours, the silent ones included.

### The same filter written with query

`query` takes the condition as text, with `and`, `or` and `not` allowed because pandas reads the
text itself, and the column names written bare:

In [12]:
print(len(readings.query("station == 'Oslo' and celsius > 15")), "warm hours at Oslo")

limit = 18
print(len(readings.query("celsius > @limit")), "hours above", limit)

1043 warm hours at Oslo
696 hours above 18


The first count matches the `&` version's 1,043. `@limit` reads the Python variable `limit` into the
text. `query` is shorter for a condition on two or three columns. The square-bracket form works
everywhere, including with masks you built earlier, which is why the rest of this guide uses it.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions
notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/08-boolean-filtering-solutions.ipynb).

**1.** Count the hours Tromso was warmer than 10 degrees.

In [13]:
# your code here

**2.** Show Svalbard's readings above 5 degrees, and print how many there are.

In [14]:
# your code here

**3.** Count the hours at Bergen or Oslo below -5 degrees, using `isin` and `&`, and say which
station they came from.

In [15]:
# your code here

**4.** Count each station's hours between 0 and 1 degree, both ends included, with `between` and
`value_counts`.

In [16]:
# your code here

**5.** Use `query` to find Oslo's hours above 18 degrees, and print how many there are.

In [17]:
# your code here

## Common errors

### ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [18]:
readings[(readings["station"] == "Oslo") and (readings["celsius"] > 15)]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

`and` asks whether its left side is true as a whole, and a Series of 35,040 values has no single
answer, so pandas raises instead of guessing. The names in the message each reduce a Series to one
value, such as `.any()`, which is `True` if any value is, and `.bool()` among them is deprecated in
pandas 2.2 and warns when called. For a row-by-row and, use `&`.

The same message comes from a line that uses `&` and leaves out the parentheses:

In [19]:
readings[readings["celsius"] > 15 & readings["time"].dt.hour < 12]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

`&` binds more tightly than `>` and `<`, so Python first worked out `15 & readings["time"].dt.hour`,
which is a Series of whole numbers. That left `readings["celsius"] > ... < 12`, and Python reads two
comparisons in a row as a chained comparison, `(a > b) and (b < 12)`, whose `and` raised as it did
in the cell above. Put each condition in parentheses:
`(readings["celsius"] > 15) & (readings["time"].dt.hour < 12)`.

### TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]

In [20]:
readings[readings["celsius"] > 15.5 & readings["celsius"] < 18]

TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]

This is the same missing pair of parentheses, and this time the column to the right of `&` holds
decimals. Python tried `15.5 & readings["celsius"]` first. `&` is defined for whole numbers and for
`True` and `False`, and the readings are `float64`, so it raised before any comparison ran. pandas
turns the number into `True` before its last try, which is why the message names a scalar of type
`bool`, and `rand_` is the name of the reversed `&` that it tried. The message says nothing about
parentheses, and parentheses are the fix:
`(readings["celsius"] > 15.5) & (readings["celsius"] < 18)`.

### No error, and two filters that ask the same question disagree

In [21]:
at_or_above = readings["celsius"] >= 0
not_below = ~(readings["celsius"] < 0)

print("celsius >= 0:   ", at_or_above.sum())
print("~(celsius < 0): ", not_below.sum())
print("rows in one and not the other:", (not_below & ~at_or_above).sum())

celsius >= 0:    22874
~(celsius < 0):  22898
rows in one and not the other: 24


The two masks read as the same question, and they differ by Svalbard's 24 silent hours. `NaN >= 0`
is `False`, so the first mask leaves them out. `NaN < 0` is also `False`, and `~` turned those into
`True`, so the second mask counts them as hours that were not below freezing. Decide what a missing
reading should do, and say so in the condition: `readings["celsius"].notna() & ...` keeps only real
readings, and the **Missing Data** notebook covers `notna`.

## Recap

- Make a mask by comparing a column with a value, and filter a table with it in square brackets
- Combine conditions with `&`, `|` and `~`, with each condition in its own parentheses
- Match a list of values with `isin` and a range with `between`
- Count the rows a mask keeps with `sum`, and the fraction it keeps with `mean`
- Write the same filter as text with `query`, and say what a missing reading should do in a
  condition

## What is next

**The Index** notebook looks at the labels a filtered table keeps: what the index is for, how to set
your own, and how a station and a time can label a row together.

---

&#8592; **Previous:** [loc and iloc](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/07-loc-and-iloc.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [The Index](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/09-the-index.ipynb) &#8594;
